# 01 — Baseline classifier (ResNet18)

**Project:** Quantitatively Audited Explainable AI for Brain Tumor MRI Classification

This notebook trains the model that every later notebook explains, audits and stress-tests:
an ImageNet-pretrained ResNet18 with a `Dropout(0.3) → Linear(512, 3)` head, trained on the
patient-level split from notebook 0.

**Inputs.** Attach notebook 0's committed output (**Add Input → Your Work → Notebooks**). This
notebook finds it under `/kaggle/input/`. If it isn't attached, a fallback cell rebuilds the data
here using the same split procedure and seed (slower, and it adds about 1 GB of `.mat` files to
this notebook's output).

**Outputs** (under `/kaggle/working/`):

```
results/checkpoints/best_model.pt   # weights + model cfg + class names + val metrics (for notebook 2)
results/history.csv                 # per-epoch train/val loss, acc, lr
results/test_metrics.json           # final held-out numbers
results/test_predictions.csv        # per-image probabilities (notebook 5 needs these)
results/figures/*.png               # curves + confusion matrix, for the write-up
configs/baseline_resnet18.yaml      # merged config actually used for this run
src/                                # notebook 0's modules + dataset, transforms, classifier
```

**Accelerator must be GPU.** The first code cell stops the notebook if it isn't.

## 1. GPU check

The first cell stops with an error if there's no GPU. Thirty epochs of ResNet18 on CPU would take
hours and only fail at the end, so it's better to find out in the first second.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Set Session options -> Accelerator -> GPU, then restart the session.")

DEVICE = torch.device("cuda")
print("CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0), f"({torch.cuda.device_count()} visible)")
print("torch", torch.__version__, "| CUDA", torch.version.cuda)

In [ ]:
import copy
import hashlib
import json
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torchvision
import yaml
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, f1_score)

WORK = Path("/kaggle/working")
RESULTS = WORK / "results"
CKPT_DIR = RESULTS / "checkpoints"
FIG_DIR = RESULTS / "figures"
for d in [CKPT_DIR, FIG_DIR, WORK / "configs"]:
    d.mkdir(parents=True, exist_ok=True)
print("torchvision", torchvision.__version__)

## 2. Locate notebook 0's output

Kaggle mounts an attached notebook's output read-only under `/kaggle/input/`. The exact folder
name depends on the notebook slug (and Kaggle has changed the nesting before), so I search for
`manifest.csv` a few levels down. I accept a candidate only if the first `file_path` in it
actually resolves to a `.mat` file. This also checks that the data came along with the manifest.

`/kaggle/working/` is checked too, so after the fallback has run once, re-running this cell in
the same session doesn't rebuild everything.

In [ ]:
def find_nb0_output():
    candidates = [WORK / "manifest.csv"]
    for depth in ["*", "*/*", "*/*/*", "*/*/*/*"]:
        candidates += sorted(Path("/kaggle/input").glob(f"{depth}/manifest.csv"))
    for m in candidates:
        if not m.exists():
            continue
        first = pd.read_csv(m, nrows=1).file_path.iloc[0]
        # notebook 0 stores paths relative to <output>/data; tolerate a results/ subfolder too
        for data_root in [m.parent / "data", m.parent.parent / "data"]:
            if (data_root / first).exists():
                return m, data_root
    return None, None


MANIFEST_PATH, DATA_ROOT = find_nb0_output()
NB0_DIR = DATA_ROOT.parent if DATA_ROOT is not None else None
print("manifest :", MANIFEST_PATH)
print("data root:", DATA_ROOT)

### 2a. Fallback: rebuild the data here

This only runs if notebook 0's output wasn't found. It's a condensed copy of notebook 0's
download → extract → parse → split, with the **same seed (42), the same two-stage stratified
split and the same patient ordering**, so it should give the identical split. The cell prints the
split counts so you can compare them with notebook 0's `run_info.json`.

Attaching notebook 0 is still the better option: then the split comes from one recorded run, and
there's no second copy of the code that could drift.

In [ ]:
if MANIFEST_PATH is None:
    import zipfile
    import requests
    from sklearn.model_selection import train_test_split
    from tqdm.auto import tqdm

    print("notebook 0 output not attached -- rebuilding data in /kaggle/working")
    for pkg in ["src", "src/data", "src/utils"]:
        (WORK / pkg).mkdir(parents=True, exist_ok=True)
        (WORK / pkg / "__init__.py").touch()

    # keep these two in sync with notebook 0's %%writefile cells
    (WORK / "src/utils/seed.py").write_text("""import os
import random

import numpy as np
import torch


def set_seed(seed: int = 42, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
""")
    (WORK / "src/data/parse_mat.py").write_text("""from dataclasses import dataclass

import h5py
import numpy as np

LABEL_NAMES = {1: "meningioma", 2: "glioma", 3: "pituitary"}


@dataclass
class TumorSample:
    label: int
    patient_id: str
    image: np.ndarray
    tumor_mask: np.ndarray
    tumor_border: np.ndarray


def _decode_chars(ds) -> str:
    return "".join(chr(int(c)) for c in np.asarray(ds).ravel())


def read_mat(path) -> TumorSample:
    with h5py.File(path, "r") as f:
        cj = f["cjdata"]
        image = np.asarray(cj["image"]).T
        mask = np.asarray(cj["tumorMask"]).T.astype(np.uint8)
        return TumorSample(
            label=int(np.asarray(cj["label"]).item()),
            patient_id=_decode_chars(cj["PID"]).strip(),
            image=image,
            tumor_mask=mask,
            tumor_border=np.asarray(cj["tumorBorder"]).ravel().reshape(-1, 2),
        )
""")
    if str(WORK) not in sys.path:
        sys.path.insert(0, str(WORK))
    from src.data.parse_mat import read_mat, LABEL_NAMES

    raw_dir, mat_dir = WORK / "data/raw", WORK / "data/mat"
    raw_dir.mkdir(parents=True, exist_ok=True)
    mat_dir.mkdir(parents=True, exist_ok=True)
    headers = {"User-Agent": "Mozilla/5.0 (research download; brain-tumor-xai-audit)"}
    files = requests.get("https://api.figshare.com/v2/articles/1512427/files", timeout=30, headers=headers).json()
    for f in files:
        dest = raw_dir / f["name"]
        if dest.exists() or not f["name"].endswith(".zip"):
            continue
        with requests.get(f["download_url"], stream=True, timeout=60, headers=headers) as r:
            r.raise_for_status()
            with open(dest, "wb") as fh:
                for block in tqdm(r.iter_content(1 << 20), desc=f["name"], total=f["size"] >> 20):
                    fh.write(block)
    for zp in sorted(raw_dir.glob("*.zip")):
        with zipfile.ZipFile(zp) as zf:
            for m in zf.infolist():
                if m.filename.endswith(".mat"):
                    with zf.open(m) as src, open(mat_dir / Path(m.filename).name, "wb") as dst:
                        shutil.copyfileobj(src, dst)
        zp.unlink()

    mat_files = sorted(mat_dir.glob("*.mat"), key=lambda p: int(p.stem))
    assert len(mat_files) == 3064, len(mat_files)
    rows = []
    for p in tqdm(mat_files, desc="parsing"):
        s = read_mat(p)
        rows.append({"file_path": str(p.relative_to(WORK / "data")), "label": s.label,
                     "label_name": LABEL_NAMES[s.label], "patient_id": s.patient_id,
                     "tumor_pixel_count": int(s.tumor_mask.sum()),
                     "image_h": s.image.shape[0], "image_w": s.image.shape[1]})
    mf = pd.DataFrame(rows)
    patients = mf.groupby("patient_id").agg(label=("label", "first")).reset_index()
    train_p, rest_p = train_test_split(patients, test_size=0.30, stratify=patients.label, random_state=42)
    val_p, test_p = train_test_split(rest_p, test_size=0.50, stratify=rest_p.label, random_state=42)
    split_of = {pid: n for n, part in [("train", train_p), ("val", val_p), ("test", test_p)] for pid in part.patient_id}
    mf["split"] = mf.patient_id.map(split_of)
    mf.to_csv(WORK / "manifest.csv", index=False)
    print(mf.groupby("split").agg(images=("file_path", "size"), patients=("patient_id", "nunique")))

    MANIFEST_PATH, DATA_ROOT = find_nb0_output()
    NB0_DIR = DATA_ROOT.parent
else:
    print("using notebook 0 output at", NB0_DIR)

### 2b. Bring notebook 0's code and config into this notebook's output

I copy notebook 0's `src/` and config into `/kaggle/working/` and import only from there, rather
than adding `/kaggle/input/<nb0>/` to `sys.path`. The reason: `src` is a normal package (it has
`__init__.py`), and Python doesn't merge a package across two `sys.path` entries. If notebook 2
had both notebook 0's and notebook 1's `src/` on the path, whichever came first would hide the
other, and `src.models.classifier` would silently not exist. After this cell, notebook 1's output
holds the **complete** `src/` tree, and later notebooks only need the newest one on their path.

In [ ]:
if NB0_DIR != WORK:
    shutil.copytree(NB0_DIR / "src", WORK / "src", dirs_exist_ok=True)
    nb0_cfg = NB0_DIR / "configs" / "baseline_resnet18.yaml"
else:
    nb0_cfg = WORK / "configs" / "baseline_resnet18.yaml"   # absent in the fallback path

for pkg in ["src", "src/data", "src/utils", "src/models"]:
    (WORK / pkg).mkdir(parents=True, exist_ok=True)
    (WORK / pkg / "__init__.py").touch()
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

MANIFEST_MD5 = hashlib.md5(MANIFEST_PATH.read_bytes()).hexdigest()
print("manifest md5:", MANIFEST_MD5)   # stored in the checkpoint so notebook 2+ can confirm the split

## 3. Config

Notebook 0's YAML has the shared values (seed, image size, batch size, dropout, lr, epochs, and
the audit and uncertainty settings). The training details below (weight decay, cosine schedule,
early-stopping patience, augmentation strengths) weren't in that file, so I add them as
**defaults**. Any value already in notebook 0's YAML wins. If you paste your full local
`baseline_resnet18.yaml` into notebook 0, those values take precedence automatically.

The merged result is written back to `/kaggle/working/configs/`, so this notebook's output records
exactly what was used for this checkpoint.

In [ ]:
DEFAULT_CFG = {
    "seed": 42,
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 4,     # Kaggle GPU sessions have 4 vCPUs; .mat decoding happens in the workers
    "data": {"class_names": ["meningioma", "glioma", "pituitary"]},
    "model": {"architecture": "resnet18", "pretrained": True, "dropout_p": 0.3},
    "train": {"optimizer": "adam", "lr": 3e-4, "weight_decay": 1e-4, "scheduler": "cosine",
              "epochs": 30, "early_stopping_patience": 7, "amp": True},
    "augment": {"hflip_p": 0.5, "rotation_deg": 10, "brightness": 0.1},
    "explain": {"methods": ["gradcam", "integrated_gradients"], "target_layer": "layer4"},
    "audit": {"deletion_insertion_steps": 20,
              "stability_transforms": [{"type": "rotate", "degrees": 5},
                                       {"type": "rotate", "degrees": -5},
                                       {"type": "brightness", "factor": 1.10}],
              "localization_threshold": 0.5},
    "uncertainty": {"n_forward_passes": 30},
}


def merge(defaults, override):
    out = copy.deepcopy(defaults)
    for k, v in (override or {}).items():
        out[k] = merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out


loaded = yaml.safe_load(nb0_cfg.read_text()) if nb0_cfg.exists() else {}
print("loaded notebook 0 config" if loaded else "no notebook 0 config found, using defaults only")
CFG = merge(DEFAULT_CFG, loaded)

with open(WORK / "configs" / "baseline_resnet18.yaml", "w") as f:
    yaml.safe_dump(CFG, f, sort_keys=False)

SEED = CFG["seed"]
CLASS_NAMES = CFG["data"]["class_names"]
print(yaml.safe_dump({k: CFG[k] for k in ["seed", "image_size", "batch_size", "model", "train", "augment"]}, sort_keys=False))

In [ ]:
from src.utils.seed import set_seed

set_seed(SEED)

## 4. Transforms

I use `torchvision.transforms.v2`, which applies one random draw to the image and its mask
together when they come as a dict of `tv_tensors`. Notebook 1 doesn't need masks, but notebooks
3, 6 and 7 do, and the mask has to stay geometrically aligned with the augmented image. Setting
this up now means I won't have to rewrite the transforms later.

- **Train:** resize to 224 → horizontal flip (p=0.5) → rotation ±10° → brightness ±10%. Axial
  brain slices are roughly left-right symmetric, so a horizontal flip produces realistic images.
  A vertical flip would put the skull base at the top, which never happens in this data, so I
  leave it out.
- **Eval:** resize + normalize only, with no randomness, so val/test numbers are reproducible.
- **Normalisation:** ImageNet mean/std, because the backbone was pretrained with those statistics.
  Masks are skipped by `Normalize`/`ColorJitter` and resized with nearest-neighbour, so they stay
  binary.
- **Stability transforms** (used in notebook 3): fixed rotations and a fixed brightness factor,
  written as degenerate ranges (`degrees=(5, 5)`) so they go through the same mask-aware code path.

In [ ]:
%%writefile /kaggle/working/src/data/transforms.py
import torch
from torchvision.transforms import v2

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def _finish():
    # ToDtype with a bare dtype only touches images, so masks stay uint8
    return [v2.ToDtype(torch.float32, scale=True), v2.Normalize(IMAGENET_MEAN, IMAGENET_STD)]


def get_train_transforms(image_size=224, hflip_p=0.5, rotation_deg=10, brightness=0.1):
    # resize first: rotating a 512px image only to downsample it afterwards is wasted work
    return v2.Compose([
        v2.Resize((image_size, image_size), antialias=True),
        v2.RandomHorizontalFlip(p=hflip_p),
        v2.RandomRotation(degrees=rotation_deg),
        v2.ColorJitter(brightness=brightness),
        *_finish(),
    ])


def get_eval_transforms(image_size=224):
    return v2.Compose([v2.Resize((image_size, image_size), antialias=True), *_finish()])


def get_stability_transform(spec, image_size=224):
    # spec is one entry of cfg["audit"]["stability_transforms"]
    if spec["type"] == "rotate":
        t = v2.RandomRotation(degrees=(spec["degrees"], spec["degrees"]))
    elif spec["type"] == "brightness":
        t = v2.ColorJitter(brightness=(spec["factor"], spec["factor"]))
    else:
        raise ValueError(spec)
    return v2.Compose([v2.Resize((image_size, image_size), antialias=True), t, *_finish()])

## 5. Dataset

Each item is read from its `.mat` file on demand. All 3064 images at 512×512 int16 would need
about 1.6 GB of RAM if cached up front. Reading on demand avoids that, and with 4 workers the h5py
decoding stays ahead of a ResNet18 forward/backward pass.

Labels map as `class_id = raw_label - 1`, which gives `[meningioma, glioma, pituitary]`. That's
the raw label order, not alphabetical. The constructor asserts that this agrees with `label_name`,
so the class order can't get mixed up silently.

In [ ]:
%%writefile /kaggle/working/src/data/dataset.py
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from torchvision import tv_tensors

from src.data.parse_mat import read_mat

CLASS_NAMES = ["meningioma", "glioma", "pituitary"]


def minmax_uint8(img):
    # per-image, because intensity scales differ between scanners/acquisitions
    img = img.astype(np.float32)
    lo, hi = img.min(), img.max()
    return ((img - lo) / (hi - lo + 1e-8) * 255).astype(np.uint8)


class BrainTumorDataset(Dataset):
    def __init__(self, manifest, data_root, split=None, transform=None, return_mask=False):
        df = manifest if isinstance(manifest, pd.DataFrame) else pd.read_csv(manifest)
        if split is not None:
            df = df[df.split == split]
        self.df = df.reset_index(drop=True)
        self.data_root = Path(data_root)
        self.transform = transform
        self.return_mask = return_mask
        assert all(CLASS_NAMES[l - 1] == n for l, n in zip(self.df.label, self.df.label_name))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        s = read_mat(self.data_root / r.file_path)
        img = torch.from_numpy(minmax_uint8(s.image)).unsqueeze(0).repeat(3, 1, 1)
        sample = {"image": tv_tensors.Image(img)}
        if self.return_mask:
            sample["tumor_mask"] = tv_tensors.Mask(torch.from_numpy(s.tumor_mask).unsqueeze(0))
        if self.transform is not None:
            sample = self.transform(sample)
        if self.return_mask:
            sample["tumor_mask"] = sample["tumor_mask"][0]
        sample["label"] = int(r.label) - 1
        sample["patient_id"] = r.patient_id
        sample["file_path"] = r.file_path
        return sample

## 6. Model

`torchvision`'s ResNet18 with ImageNet weights, with `fc` replaced by `Dropout(p) → Linear(512, 3)`.

The dropout in the head is what notebook 4 relies on. MC-dropout keeps that one layer active at
test time and samples 30 forward passes, so the dropout has to be **part of the trained network**:
the weights must have been learned with units being dropped. A head that never had dropout can't
have it added afterwards and still give meaningful uncertainty.

I keep the model as a plain torchvision `ResNet` and don't wrap it, so `model.layer4`, which is
what `explain.target_layer` names, resolves directly for Grad-CAM in notebook 2. The dropout layer
is `model.fc[0]`.

`load_checkpoint` builds the model with `pretrained=False` before loading the state dict, so
notebook 2 doesn't need to download ImageNet weights just to overwrite them.

In [ ]:
%%writefile /kaggle/working/src/models/classifier.py
import torch
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights


def build_model(num_classes=3, dropout_p=0.3, pretrained=True, architecture="resnet18"):
    assert architecture == "resnet18", architecture
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    model.fc = nn.Sequential(nn.Dropout(p=dropout_p), nn.Linear(model.fc.in_features, num_classes))
    return model


def load_checkpoint(path, device="cpu"):
    ckpt = torch.load(path, map_location=device, weights_only=True)
    mc = ckpt["model_cfg"]
    model = build_model(num_classes=mc["num_classes"], dropout_p=mc["dropout_p"],
                        pretrained=False, architecture=mc["architecture"])
    model.load_state_dict(ckpt["state_dict"])
    return model.to(device).eval(), ckpt

## 7. Data loaders

The train loader gets a seeded `torch.Generator`. That fixes the shuffle order and the base seed
each worker derives its augmentation RNG from, so a rerun with the same seed gets the same batches
and the same augmentations.

`drop_last=True` only for train. With 32 per batch, the last train batch could be tiny, and one
noisy BatchNorm update per epoch isn't worth keeping. Val and test keep every image.

In [ ]:
from src.data.dataset import BrainTumorDataset
from src.data.transforms import get_train_transforms, get_eval_transforms
from src.models.classifier import build_model, load_checkpoint

manifest = pd.read_csv(MANIFEST_PATH)
aug = CFG["augment"]
train_tf = get_train_transforms(CFG["image_size"], aug["hflip_p"], aug["rotation_deg"], aug["brightness"])
eval_tf = get_eval_transforms(CFG["image_size"])

datasets = {
    "train": BrainTumorDataset(manifest, DATA_ROOT, "train", train_tf),
    "val": BrainTumorDataset(manifest, DATA_ROOT, "val", eval_tf),
    "test": BrainTumorDataset(manifest, DATA_ROOT, "test", eval_tf),
}


def make_loader(split, shuffle):
    return DataLoader(
        datasets[split], batch_size=CFG["batch_size"], shuffle=shuffle, drop_last=shuffle,
        num_workers=CFG["num_workers"], pin_memory=True, persistent_workers=True,
        generator=torch.Generator().manual_seed(SEED) if shuffle else None,
    )


loaders = {"train": make_loader("train", True), "val": make_loader("val", False), "test": make_loader("test", False)}

for split, ds in datasets.items():
    counts = ds.df.label_name.value_counts().reindex(CLASS_NAMES).tolist()
    print(f"{split:5s} {len(ds):5d} images  {ds.df.patient_id.nunique():3d} patients  {dict(zip(CLASS_NAMES, counts))}")

A quick look at one augmented training batch, after normalisation and then de-normalised for
display. If these look washed out, inverted, or the rotations look much bigger than ±10°, the
transform pipeline is wrong, and it's far cheaper to see that now than after 30 epochs.

In [ ]:
from src.data.transforms import IMAGENET_MEAN, IMAGENET_STD

batch = next(iter(loaders["train"]))
print("image batch:", tuple(batch["image"].shape), batch["image"].dtype,
      f"mean {batch['image'].mean():.3f} std {batch['image'].std():.3f}")

mean, std = torch.tensor(IMAGENET_MEAN)[:, None, None], torch.tensor(IMAGENET_STD)[:, None, None]
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, img, lab in zip(axes.ravel(), batch["image"], batch["label"]):
    ax.imshow((img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
    ax.set_title(CLASS_NAMES[lab], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

set_seed(SEED)   # the preview consumed RNG; reset so training starts from a clean seeded state

## 8. Training

- **Loss:** plain `CrossEntropyLoss`, without class weights. The imbalance (glioma is about 47% of
  the images) is moderate, and balanced accuracy and per-class recall on test will show whether it
  hurts the minority class. I'd rather keep the baseline simple and *measure* the imbalance effect
  than correct for it by default.
- **Optimiser:** Adam, lr 3e-4, weight decay 1e-4. All layers are fine-tuned from the start; I'm
  not freezing the backbone first. MRI looks nothing like ImageNet, so the early layers need to adapt.
- **Schedule:** cosine annealing over `epochs`, stepped once per epoch. If early stopping fires
  first, the LR never reaches its minimum. That's acceptable, because the best-val checkpoint is
  what we keep anyway.
- **Early stopping:** on **val loss**, patience 7. Loss rather than accuracy, because accuracy on
  ~460 val images moves in coarse steps and plateaus early, while loss keeps showing overconfidence
  creeping in. Calibration matters for notebook 4, so that's the signal I want to stop on.
- **Checkpoint:** saved whenever val loss improves, so the file on disk always holds the best epoch,
  not the last one.
- **AMP (fp16 autocast):** about 2× faster on a T4. The loss scaler handles the small gradients.

"Train acc" in the log is measured with dropout and augmentation on, so it will sit below val acc
early on. That's expected and doesn't indicate a bug.

In [ ]:
model = build_model(num_classes=len(CLASS_NAMES), dropout_p=CFG["model"]["dropout_p"],
                    pretrained=CFG["model"]["pretrained"]).to(DEVICE)
print(model.fc)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

tcfg = CFG["train"]
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=tcfg["lr"], weight_decay=tcfg["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=tcfg["epochs"])
USE_AMP = tcfg["amp"]
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

In [ ]:
def run_epoch(model, loader, train):
    model.train(train)
    total_loss, labels, probs = 0.0, [], []
    with torch.set_grad_enabled(train):
        for batch in loader:
            x = batch["image"].to(DEVICE, non_blocking=True)
            y = batch["label"].to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * len(y)
            labels.append(y.cpu())
            probs.append(logits.float().softmax(1).detach().cpu())
    labels = torch.cat(labels).numpy()
    probs = torch.cat(probs).numpy()
    preds = probs.argmax(1)
    return {"loss": total_loss / len(labels), "acc": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro"),
            "labels": labels, "preds": preds, "probs": probs}

In [ ]:
CKPT_PATH = CKPT_DIR / "best_model.pt"
history, best = [], {"loss": float("inf"), "epoch": -1}
bad_epochs = 0

for epoch in range(1, tcfg["epochs"] + 1):
    t0 = time.time()
    lr = optimizer.param_groups[0]["lr"]
    tr = run_epoch(model, loaders["train"], train=True)
    va = run_epoch(model, loaders["val"], train=False)
    scheduler.step()

    history.append({"epoch": epoch, "lr": lr, "train_loss": tr["loss"], "train_acc": tr["acc"],
                    "val_loss": va["loss"], "val_acc": va["acc"], "val_macro_f1": va["macro_f1"]})

    # tiny margin so float noise in the 4th decimal doesn't count as "improving"
    improved = va["loss"] < best["loss"] - 1e-4
    if improved:
        best = {"loss": va["loss"], "acc": va["acc"], "macro_f1": va["macro_f1"], "epoch": epoch}
        bad_epochs = 0
        torch.save({
            "state_dict": model.state_dict(),
            "model_cfg": {"architecture": CFG["model"]["architecture"], "num_classes": len(CLASS_NAMES),
                          "dropout_p": CFG["model"]["dropout_p"], "image_size": CFG["image_size"]},
            "class_names": CLASS_NAMES,
            "val_metrics": {k: float(v) for k, v in best.items()},
            "config": CFG,
            "manifest_md5": MANIFEST_MD5,
            "seed": SEED,
        }, CKPT_PATH)
    else:
        bad_epochs += 1

    print(f"ep {epoch:2d} | lr {lr:.2e} | train {tr['loss']:.4f} / {tr['acc']:.3f} | "
          f"val {va['loss']:.4f} / {va['acc']:.3f} f1 {va['macro_f1']:.3f} | "
          f"{time.time() - t0:5.1f}s {'*' if improved else ''}")

    if bad_epochs >= tcfg["early_stopping_patience"]:
        print(f"early stop: no val-loss improvement for {bad_epochs} epochs")
        break

print(f"\nbest epoch {best['epoch']}: val loss {best['loss']:.4f}, acc {best['acc']:.3f}, macro-F1 {best['macro_f1']:.3f}")
hist = pd.DataFrame(history)
hist.to_csv(RESULTS / "history.csv", index=False)

### Training curves

The dashed line marks the checkpoint that was kept. Things to check: the train/val gap (a gap that
keeps widening after the best epoch is the overfitting early stopping exists to catch), and whether
val loss rises while val accuracy stays flat. That pattern means the model keeps getting more
confident without getting more correct. It's the miscalibration notebook 4 will measure directly.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist.epoch, hist.train_loss, label="train")
ax1.plot(hist.epoch, hist.val_loss, label="val")
ax1.set_ylabel("cross-entropy loss")
ax2.plot(hist.epoch, hist.train_acc, label="train (dropout + aug on)")
ax2.plot(hist.epoch, hist.val_acc, label="val")
ax2.set_ylabel("accuracy")
for ax in (ax1, ax2):
    ax.axvline(best["epoch"], color="gray", ls="--", lw=1)
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Test evaluation

The test split is used **once**, here, after all training decisions are final. Nothing above
(epochs, patience, augmentation) was tuned on it. If I later change something because of these
numbers, the test set is no longer an honest held-out estimate, and the write-up should say so.

I reload the best checkpoint from disk with `load_checkpoint`, not the in-memory model, which holds
the *last* epoch's weights. That also exercises the exact loading path notebook 2 will use, so a
broken checkpoint shows up here and not in the next notebook.

Besides plain accuracy I report **balanced accuracy** and **macro-F1**. With glioma at roughly half
the data, a model that's weak on meningioma can still reach a respectable plain accuracy, and the
per-class recall exposes that.

In [ ]:
model, ckpt = load_checkpoint(CKPT_PATH, DEVICE)
assert ckpt["manifest_md5"] == MANIFEST_MD5
print(f"loaded epoch {ckpt['val_metrics']['epoch']:.0f} checkpoint, val acc {ckpt['val_metrics']['acc']:.3f}")

te = run_epoch(model, loaders["test"], train=False)   # train=False -> model.eval(), dropout off
y, yhat = te["labels"], te["preds"]

print(f"\nTEST  n={len(y)}  patients={datasets['test'].df.patient_id.nunique()}")
print(f"accuracy          {accuracy_score(y, yhat):.4f}")
print(f"balanced accuracy {balanced_accuracy_score(y, yhat):.4f}")
print(f"macro-F1          {f1_score(y, yhat, average='macro'):.4f}\n")
print(classification_report(y, yhat, target_names=CLASS_NAMES, digits=3))

In [ ]:
cm = confusion_matrix(y, yhat)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("test confusion matrix (counts)")
ConfusionMatrixDisplay(cm / cm.sum(1, keepdims=True), display_labels=CLASS_NAMES).plot(
    ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f")
axes[1].set_title("row-normalised (= per-class recall on the diagonal)")
for ax in axes:
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
plt.tight_layout()
fig.savefig(FIG_DIR / "confusion_matrix_test.png", dpi=150, bbox_inches="tight")
plt.show()

Saving per-image test predictions as well as the summary: notebook 5 (confidence × faithfulness,
failure cases) needs to know *which* images were wrong and how confident the model was on each.
Recomputing that there would mean another inference pass and another chance for drift.

In [ ]:
test_df = datasets["test"].df[["file_path", "patient_id", "label_name"]].copy()
test_df["true"] = y
test_df["pred"] = yhat
test_df["pred_name"] = [CLASS_NAMES[i] for i in yhat]
for i, c in enumerate(CLASS_NAMES):
    test_df[f"p_{c}"] = te["probs"][:, i]
test_df["confidence"] = te["probs"].max(1)
test_df["correct"] = test_df.true == test_df.pred
test_df.to_csv(RESULTS / "test_predictions.csv", index=False)

report = classification_report(y, yhat, target_names=CLASS_NAMES, digits=4, output_dict=True)
test_metrics = {
    "n_test_images": int(len(y)),
    "n_test_patients": int(datasets["test"].df.patient_id.nunique()),
    "accuracy": float(accuracy_score(y, yhat)),
    "balanced_accuracy": float(balanced_accuracy_score(y, yhat)),
    "macro_f1": float(f1_score(y, yhat, average="macro")),
    "test_loss": float(te["loss"]),
    "per_class": {c: {k: float(v) for k, v in report[c].items()} for c in CLASS_NAMES},
    "confusion_matrix": cm.tolist(),
    "class_names": CLASS_NAMES,
    "best_epoch": int(best["epoch"]),
    "epochs_run": int(len(hist)),
    "val_metrics_at_best": ckpt["val_metrics"],
    "manifest_md5": MANIFEST_MD5,
    "checkpoint": str(CKPT_PATH.relative_to(WORK)),
    "gpu": torch.cuda.get_device_name(0),
    "versions": {"torch": torch.__version__, "torchvision": torchvision.__version__},
}
with open(RESULTS / "test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

print("misclassified:", int((~test_df.correct).sum()), "of", len(test_df))
display(test_df.loc[~test_df.correct].sort_values("confidence", ascending=False).head(10))

In [ ]:
!ls -la /kaggle/working/results /kaggle/working/results/checkpoints /kaggle/working/results/figures
!du -sh /kaggle/working/results/checkpoints/best_model.pt

## Closing checklist

**Is the result plausible?**

- [ ] **Test accuracy roughly 0.90–0.96.** That's the usual range for a pretrained ResNet18 on this
  dataset with a *patient-level* split. Papers reporting 98–99% mostly split by image, which leaks
  near-duplicate slices across splits.
  - Above ~0.98: be suspicious first. Re-check `leakage_check.json` from notebook 0, and make sure
    the manifest md5 printed above matches the one in notebook 0's run.
  - Below ~0.85: more likely a bug than a weak model. Check the batch preview, the class mapping,
    and that the loaded checkpoint is the best epoch.
- [ ] **Meningioma recall is the lowest.** That's typical: meningiomas vary most in where they sit
  and overlap visually with both other classes. If glioma recall is the lowest instead, look at the
  confusion matrix before moving on.
- [ ] **Early stopping fired** (or val loss flattened) before epoch 30, and the best epoch isn't
  epoch 1–2. A best epoch that early would point to an LR or augmentation problem.
- [ ] The test/val gap is a few points at most. With only ~35 patients in each split, a gap of
  2–3 points can be pure patient sampling noise. Keep that in mind before over-interpreting
  per-class differences in later notebooks.

**Persisting the checkpoint for notebook 2:**

1. Click **Save Version → Save & Run All (Commit) → Save**. This re-runs the whole notebook in a
   fresh session (about 15–25 min on a T4, depending on where early stopping lands). Keep the
   accelerator **on GPU** for this commit; the first cell refuses to run without one.
2. When the version finishes, open its **Output** tab and confirm that
   `results/checkpoints/best_model.pt`, `results/test_metrics.json`, `results/test_predictions.csv`
   and `src/models/classifier.py` are present.
3. In notebook 2, attach **both** notebook 0's output (for the `.mat` data) and this notebook's
   output (checkpoint + `src/`). Put only **this** notebook's directory on `sys.path` (see 2b for
   why), then:
   ```python
   from src.models.classifier import load_checkpoint
   model, ckpt = load_checkpoint(NB1_DIR / "results/checkpoints/best_model.pt", DEVICE)
   ```
4. Note: the committed run's numbers can differ slightly from this interactive run. cuDNN
   determinism is on, but AMP and GPU type (T4 vs P100) can still move results in the third decimal.
   **Quote the committed version's `test_metrics.json` in the write-up**, because that's the run
   the saved checkpoint comes from.